# ResNet34 Supervised Baseline for Culvert Blockage Classification

This notebook implements the ResNet34 supervised reference experiment used to
assess whether a conventional ImageNet-pretrained convolutional neural network
can classify culvert trash-screen conditions and transfer to previously unseen
monitoring sites.

The model receives cropped CCTV images and predicts one of two classes:
**clear** or **blocked**. The convolutional ResNet34 backbone is retained as a
fixed pretrained feature extractor and only the final fully connected
classification layer is trained on the culvert dataset.

## Experimental design

The ten monitoring sites are separated by camera location:

- **8 development sites** are available for model development.
- Of these, **6 sites are used for training** and **2 sites for validation**.
- **2 sites, Cornwall Bude Cedar Grove and Brutondam, are held out completely**
  until the final cross-site evaluation.

This site-level separation is important because images captured by the same
fixed CCTV camera share background, geometry, viewpoint and infrastructure
characteristics. Random image-level splitting would therefore risk measuring
recognition of site-specific visual cues rather than transfer to a new
monitoring environment.

## Reproducibility and repository structure

The public notebook uses repository-relative paths rather than Google Drive or
machine-specific directories. Expected inputs are:

```text
data/
├── site_split.csv
└── resnet_cropped_4000/
    ├── cropped_dataset_index.csv
    └── <cropped image files>
```

Generated artefacts are written to:

```text
outputs/resnet34_supervised/
```

Stored execution outputs have been removed from the public notebook so that no
local paths or cached execution state are exposed.


In [ ]:
# Reproducible experiment setup

import copy
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.models import ResNet34_Weights

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

RANDOM_SEED = 42


def set_seed(seed=RANDOM_SEED):
    """Set random seeds used by Python, NumPy and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Favour reproducibility over maximum GPU throughput.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


In [ ]:
# Repository-relative paths

# The notebook assumes that it is launched from the repository root.
REPO_ROOT = Path.cwd()
DATA_ROOT = REPO_ROOT / "data"

CROPPED_DATA_ROOT = DATA_ROOT / "resnet_cropped_4000"
CROPPED_INDEX_PATH = CROPPED_DATA_ROOT / "cropped_dataset_index.csv"
SPLIT_PATH = DATA_ROOT / "site_split.csv"

EXPERIMENT_FOLDER = (
    REPO_ROOT / "outputs" / "resnet34_supervised"
)

EXPERIMENT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

print("Cropped dataset:", CROPPED_INDEX_PATH)
print("Site split:", SPLIT_PATH)
print("Results folder:", EXPERIMENT_FOLDER)


In [ ]:
# Load the cropped-image index

if not CROPPED_INDEX_PATH.exists():
    raise FileNotFoundError(
        f"Cropped dataset index not found: {CROPPED_INDEX_PATH}"
    )

dataset_df = pd.read_csv(
    CROPPED_INDEX_PATH
)

required_columns = {
    "site",
    "label",
    "filename",
    "cropped_path"
}

if not required_columns.issubset(dataset_df.columns):
    raise ValueError(
        f"Dataset index must contain: {sorted(required_columns)}"
    )

if len(dataset_df) != 4000:
    raise ValueError(
        f"Expected 4,000 images but found {len(dataset_df)}."
    )

# Reconstruct portable paths from filenames rather than preserving
# machine-specific paths stored in the original index.
dataset_df["image_path"] = dataset_df["filename"].apply(
    lambda name: str(CROPPED_DATA_ROOT / name)
)

# If the repository stores site subdirectories, fall back to
# <root>/<site>/<filename>.
missing = ~dataset_df["image_path"].apply(
    lambda p: Path(p).exists()
)

if missing.any():
    dataset_df.loc[missing, "image_path"] = dataset_df.loc[
        missing
    ].apply(
        lambda row: str(
            CROPPED_DATA_ROOT
            / row["site"]
            / row["filename"]
        ),
        axis=1
    )

missing_paths = dataset_df.loc[
    ~dataset_df["image_path"].apply(
        lambda p: Path(p).exists()
    ),
    "image_path"
]

if len(missing_paths) > 0:
    raise FileNotFoundError(
        "Some cropped images could not be located beneath "
        f"{CROPPED_DATA_ROOT}. First missing path: "
        f"{missing_paths.iloc[0]}"
    )

print("Images:", len(dataset_df))
display(dataset_df.head())


In [ ]:
# Verify class labels and site coverage

valid_labels = {"clear", "blocked"}

if not set(dataset_df["label"].str.lower().unique()).issubset(
    valid_labels
):
    raise ValueError(
        "Unexpected class label found. "
        "Expected only 'clear' and 'blocked'."
    )

print("Number of monitoring sites:", dataset_df["site"].nunique())
print("\nImages by site and class:")

display(
    dataset_df.groupby(["site", "label"])
    .size()
    .unstack(fill_value=0)
)


In [ ]:
# Confirm that all repository-relative image paths resolve

path_exists = dataset_df["image_path"].apply(
    lambda p: Path(p).exists()
)

print("Images found:", int(path_exists.sum()))
print("Missing images:", int((~path_exists).sum()))

assert path_exists.all()


## 1. Cross-site data partition

The fixed split first separates the eight development sites from the two
held-out test sites. A seeded random selection then assigns two of the eight
development sites to validation and the remaining six to training.

The validation cameras are used to select the best training epoch and
classification threshold. The two test cameras are excluded from all model
selection decisions and are evaluated only after the final model has been
trained.


In [ ]:
# Load the fixed cross-site split

if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Site split file not found: {SPLIT_PATH}"
    )

split_summary = pd.read_csv(
    SPLIT_PATH
)

required_split_columns = {"site", "role"}

if not required_split_columns.issubset(
    split_summary.columns
):
    raise ValueError(
        f"site_split.csv must contain: "
        f"{sorted(required_split_columns)}"
    )

DEVELOPMENT_SITES = sorted(
    split_summary.loc[
        split_summary["role"].str.lower() == "development",
        "site"
    ].tolist()
)

TEST_SITES = sorted(
    split_summary.loc[
        split_summary["role"].str.lower() == "test",
        "site"
    ].tolist()
)

EXPECTED_TEST_SITES = {
    "Cornwall_BudeCedarGrove",
    "sites_brutondam_cam1"
}

assert len(DEVELOPMENT_SITES) == 8
assert len(TEST_SITES) == 2
assert set(DEVELOPMENT_SITES).isdisjoint(TEST_SITES)

if set(TEST_SITES) != EXPECTED_TEST_SITES:
    raise ValueError(
        "Unexpected held-out test sites. "
        f"Expected {sorted(EXPECTED_TEST_SITES)}, "
        f"found {TEST_SITES}."
    )

if set(dataset_df["site"].unique()) != set(
    DEVELOPMENT_SITES + TEST_SITES
):
    raise ValueError(
        "The sites in the image index do not match site_split.csv."
    )

print("Development sites:")
for site in DEVELOPMENT_SITES:
    print(" ", site)

print("\nHeld-out test sites:")
for site in TEST_SITES:
    print(" ", site)


In [ ]:
# Split development sites into training and validation

split_rng = np.random.default_rng(
    RANDOM_SEED
)

VALIDATION_SITES = sorted(
    split_rng.choice(
        DEVELOPMENT_SITES,
        size=2,
        replace=False
    ).tolist()
)

TRAIN_SITES = [
    site
    for site in DEVELOPMENT_SITES
    if site not in VALIDATION_SITES
]


assert len(TRAIN_SITES) == 6
assert len(VALIDATION_SITES) == 2


print("Training sites:")

for site in TRAIN_SITES:
    print(" ", site)


print("\nValidation sites:")

for site in VALIDATION_SITES:
    print(" ", site)


print("\nHeld-out test sites:")

for site in TEST_SITES:
    print(" ", site)


In [ ]:
# Create train, validation and test DataFrames

train_df = (
    dataset_df[
        dataset_df["site"].isin(
            TRAIN_SITES
        )
    ]
    .reset_index(drop=True)
)

validation_df = (
    dataset_df[
        dataset_df["site"].isin(
            VALIDATION_SITES
        )
    ]
    .reset_index(drop=True)
)

test_df = (
    dataset_df[
        dataset_df["site"].isin(
            TEST_SITES
        )
    ]
    .reset_index(drop=True)
)


print(
    "Training images:",
    len(train_df)
)

print(
    "Validation images:",
    len(validation_df)
)

print(
    "Test images:",
    len(test_df)
)


print("\nTraining classes:")
print(
    train_df["label"].value_counts()
)

print("\nValidation classes:")
print(
    validation_df["label"].value_counts()
)

print("\nTest classes:")
print(
    test_df["label"].value_counts()
)


## 2. Image preprocessing

ResNet34 uses the preprocessing associated with the selected
`IMAGENET1K_V1` pretrained weights. This applies the resize/crop and
normalisation expected by the original ImageNet-trained network.

Using the pretrained model's native preprocessing preserves compatibility
between the input distribution and the representation learned during
pretraining.


In [ ]:
# ResNet34 preprocessing

weights = ResNet34_Weights.IMAGENET1K_V1

RESNET34_TRANSFORM = weights.transforms()

print(
    RESNET34_TRANSFORM
)


In [ ]:
# Dataset class

class CulvertResNetDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform is not None:

            image = self.transform(
                image
            )

        label = (
            1
            if row["label"] == "blocked"
            else 0
        )

        return (
            image,
            label,
            row["site"],
            row["image_path"]
        )


In [ ]:
# DataLoaders

BATCH_SIZE = 32

train_dataset = CulvertResNetDataset(
    train_df,
    transform=RESNET34_TRANSFORM
)

validation_dataset = CulvertResNetDataset(
    validation_df,
    transform=RESNET34_TRANSFORM
)

test_dataset = CulvertResNetDataset(
    test_df,
    transform=RESNET34_TRANSFORM
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


## 3. Frozen-backbone transfer learning

A ResNet34 pretrained on ImageNet is used as the visual backbone. All pretrained
convolutional parameters are frozen and the original classification layer is
replaced by a new two-output fully connected layer.

Only this final layer is optimised. This design tests how useful the
pre-existing ResNet34 visual representation is for the culvert-blockage task
without allowing the convolutional backbone to adapt to the target dataset.

Batch-normalisation layers are also kept in evaluation mode during training so
that their pretrained running statistics are not updated.


In [ ]:
# Build frozen ResNet34

weights = ResNet34_Weights.IMAGENET1K_V1

resnet34_model = models.resnet34(
    weights=weights
)

for parameter in resnet34_model.parameters():
    parameter.requires_grad = False

in_features = resnet34_model.fc.in_features

resnet34_model.fc = nn.Linear(
    in_features,
    2
)

resnet34_model = resnet34_model.to(
    DEVICE
)

total_parameters = sum(
    parameter.numel()
    for parameter in resnet34_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in resnet34_model.parameters()
    if parameter.requires_grad
)

print("Model: ResNet34")
print("Weights: ImageNet pretrained")
print("Backbone: frozen")
print("Trainable layer: final fc")

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


In [ ]:
# Training and evaluation functions

def set_frozen_batchnorm_eval(
    model
):

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d
        ):
            module.eval()


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    # Keep pretrained BatchNorm statistics fixed
    set_frozen_batchnorm_eval(
        model
    )

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    for images, labels, _, _ in loader:

        images = images.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_targets.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    epoch_f1 = f1_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_f1
    )


def evaluate_model(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0
    all_targets = []
    all_predictions = []
    all_scores = []

    with torch.no_grad():

        for images, labels, _, _ in loader:

            images = images.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_targets.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_scores.extend(
                probabilities.cpu().numpy()
            )

    metrics = {

        "loss":
            running_loss
            / len(loader.dataset),

        "auc":
            roc_auc_score(
                all_targets,
                all_scores
            ),

        "accuracy":
            accuracy_score(
                all_targets,
                all_predictions
            ),

        "precision":
            precision_score(
                all_targets,
                all_predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                all_targets,
                all_predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                all_targets,
                all_predictions,
                zero_division=0
            )
    }

    return (
        metrics,
        np.array(all_targets),
        np.array(all_predictions),
        np.array(all_scores)
    )


## 4. Optimisation configuration

The trainable classification layer is optimised using AdamW with cross-entropy
loss. The learning rate and weight-decay values are fixed from the development
procedure represented in this notebook.

No additional image augmentation is applied in this experiment, so the
comparison reflects the pretrained ResNet representation plus the supervised
classification head.


In [ ]:
# ResNet34 training configuration

LEARNING_RATE = 0.001

WEIGHT_DECAY = (
    0.0003142880890840109
)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    resnet34_model.fc.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Augmentation: False")


In [ ]:
# One-epoch CPU timing test

start_time = time.time()

train_loss, train_f1 = train_one_epoch(
    resnet34_model,
    train_loader,
    criterion,
    optimizer
)

validation_metrics, _, _, _ = evaluate_model(
    resnet34_model,
    validation_loader,
    criterion
)

elapsed_seconds = (
    time.time()
    - start_time
)

print(
    "One-epoch runtime:",
    f"{elapsed_seconds / 60:.2f} minutes"
)

print(
    "Training loss:",
    f"{train_loss:.4f}"
)

print(
    "Training F1:",
    f"{train_f1:.4f}"
)

print(
    "Validation AUC:",
    f"{validation_metrics['auc']:.4f}"
)

print(
    "Validation F1:",
    f"{validation_metrics['f1']:.4f}"
)


## 5. Development training and epoch selection

A fresh ResNet34 instance is trained for the predefined development run.
Performance is evaluated after each epoch on the two validation sites.

The model state with the highest validation F1 score is retained. Selecting the
epoch using validation sites rather than the final test cameras prevents test
information from influencing training duration.


In [ ]:
# Fresh ResNet34 for the full run

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


resnet34_model = models.resnet34(
    weights=ResNet34_Weights.IMAGENET1K_V1
)

for parameter in resnet34_model.parameters():
    parameter.requires_grad = False

in_features = resnet34_model.fc.in_features

resnet34_model.fc = nn.Linear(
    in_features,
    2
)

resnet34_model = resnet34_model.to(
    DEVICE
)


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    resnet34_model.fc.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

RESNET34_EPOCHS = 19

print("Fresh ResNet34 ready")
print("Epochs:", RESNET34_EPOCHS)
print("Device:", DEVICE)


In [ ]:
# Train ResNet34

import copy

resnet34_history = []

best_validation_f1 = -np.inf
best_epoch = None
best_state = None

training_start = time.time()


for epoch in range(
    1,
    RESNET34_EPOCHS + 1
):

    epoch_start = time.time()


    # Train
    train_loss, train_f1 = train_one_epoch(
        resnet34_model,
        train_loader,
        criterion,
        optimizer
    )


    # Validate
    validation_metrics, _, _, _ = evaluate_model(
        resnet34_model,
        validation_loader,
        criterion
    )


    # Save results
    resnet34_history.append({

        "epoch": epoch,

        "train_loss":
            train_loss,

        "train_f1":
            train_f1,

        "validation_loss":
            validation_metrics["loss"],

        "validation_auc":
            validation_metrics["auc"],

        "validation_accuracy":
            validation_metrics["accuracy"],

        "validation_precision":
            validation_metrics["precision"],

        "validation_recall":
            validation_metrics["recall"],

        "validation_f1":
            validation_metrics["f1"]
    })


    # Keep best validation model
    if validation_metrics["f1"] > best_validation_f1:

        best_validation_f1 = (
            validation_metrics["f1"]
        )

        best_epoch = epoch

        best_state = copy.deepcopy(
            resnet34_model.state_dict()
        )


    # Time information
    epoch_minutes = (
        time.time()
        - epoch_start
    ) / 60


    elapsed_minutes = (
        time.time()
        - training_start
    ) / 60


    average_epoch_minutes = (
        elapsed_minutes
        / epoch
    )


    remaining_epochs = (
        RESNET34_EPOCHS
        - epoch
    )


    estimated_remaining_minutes = (
        average_epoch_minutes
        * remaining_epochs
    )


    estimated_total_minutes = (
        elapsed_minutes
        + estimated_remaining_minutes
    )


    print(
        f"\nEpoch {epoch:02d}/{RESNET34_EPOCHS}"
    )

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f}"
    )

    print(
        f"Val AUC: {validation_metrics['auc']:.4f} | "
        f"Val F1: {validation_metrics['f1']:.4f}"
    )

    print(
        f"Epoch time: {epoch_minutes:.2f} min | "
        f"Elapsed: {elapsed_minutes:.2f} min | "
        f"Estimated remaining: "
        f"{estimated_remaining_minutes:.2f} min"
    )

    print(
        f"Estimated total: "
        f"{estimated_total_minutes:.2f} min"
    )


# Convert history to DataFrame
resnet34_history_df = pd.DataFrame(
    resnet34_history
)


# Restore best validation model
resnet34_model.load_state_dict(
    best_state
)


total_training_minutes = (
    time.time()
    - training_start
) / 60


print(
    "\nResNet34 development training complete"
)

print(
    "Best validation F1:",
    f"{best_validation_f1:.4f}"
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Total training time:",
    f"{total_training_minutes:.2f} minutes"
)


In [ ]:
# Save best ResNet34 development model

RESNET34_DEV_MODEL_PATH = (
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_best_development_model.pt"
)

torch.save(
    {
        "model_state_dict":
            resnet34_model.state_dict(),

        "architecture":
            "ResNet34",

        "pretrained_weights":
            "ImageNet1K_V1",

        "optimizer":
            "AdamW",

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            WEIGHT_DECAY,

        "augmentation":
            False,

        "best_epoch":
            int(best_epoch),

        "best_validation_f1":
            float(best_validation_f1),

        "random_seed":
            RANDOM_SEED,

        "train_sites":
            TRAIN_SITES,

        "validation_sites":
            VALIDATION_SITES,

        "test_sites":
            TEST_SITES
    },

    RESNET34_DEV_MODEL_PATH
)

resnet34_history_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_development_history.csv",
    index=False
)

print("Saved best ResNet34 development model")
print(RESNET34_DEV_MODEL_PATH)


## 6. Classification-threshold selection

Although the network produces two class scores, the blocked-class softmax
probability is retained as a continuous blockage score. Candidate thresholds
between 0 and 1 are evaluated on the validation sites only.

The threshold producing the highest validation F1 score is fixed before final
testing. This separates threshold selection from the held-out evaluation and
makes the reported test metrics less optimistically biased.


In [ ]:
# Select ResNet34 classification threshold
# Threshold is selected using development validation data only.

(
    validation_metrics,
    validation_targets,
    _,
    validation_scores
) = evaluate_model(
    resnet34_model,
    validation_loader,
    criterion
)


threshold_candidates = np.linspace(
    0.0,
    1.0,
    1001
)

threshold_rows = []


for threshold in threshold_candidates:

    predictions = (
        validation_scores
        >= threshold
    ).astype(int)


    threshold_rows.append(
        {
            "threshold":
                threshold,

            "precision":
                precision_score(
                    validation_targets,
                    predictions,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    validation_targets,
                    predictions,
                    zero_division=0
                ),

            "f1":
                f1_score(
                    validation_targets,
                    predictions,
                    zero_division=0
                )
        }
    )


threshold_results_df = pd.DataFrame(
    threshold_rows
)


best_threshold_row = threshold_results_df.loc[
    threshold_results_df["f1"].idxmax()
]


RESNET34_THRESHOLD = float(
    best_threshold_row["threshold"]
)


print(
    "Selected classification threshold:",
    f"{RESNET34_THRESHOLD:.3f}"
)

print(
    "Validation F1 at selected threshold:",
    f"{best_threshold_row['f1']:.4f}"
)

print(
    "Validation precision:",
    f"{best_threshold_row['precision']:.4f}"
)

print(
    "Validation recall:",
    f"{best_threshold_row['recall']:.4f}"
)


# Save threshold search results
threshold_results_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_validation_threshold_search.csv",
    index=False
)


## 7. Final training on all development sites

Once the best epoch and decision threshold have been selected, all eight
development sites are recombined for final training. The two held-out test
sites remain untouched.

The final model is trained for the number of epochs selected during development,
thereby using all available development data without introducing the final test
sites into model fitting.


In [ ]:
# Final ResNet34 training and test data

final_train_df = (
    dataset_df[
        dataset_df["site"].isin(
            DEVELOPMENT_SITES
        )
    ]
    .reset_index(drop=True)
)

final_test_df = (
    dataset_df[
        dataset_df["site"].isin(
            TEST_SITES
        )
    ]
    .reset_index(drop=True)
)


assert len(final_train_df) == 3200
assert len(final_test_df) == 800

assert set(
    final_train_df["site"]
).isdisjoint(
    set(final_test_df["site"])
)


final_train_dataset = CulvertResNetDataset(
    final_train_df,
    transform=RESNET34_TRANSFORM
)

final_test_dataset = CulvertResNetDataset(
    final_test_df,
    transform=RESNET34_TRANSFORM
)


final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

final_test_loader = DataLoader(
    final_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


print(
    "Final training images:",
    len(final_train_df)
)

print(
    "Held-out test images:",
    len(final_test_df)
)

print(
    "\nFinal training batches:",
    len(final_train_loader)
)

print(
    "Final test batches:",
    len(final_test_loader)
)


In [ ]:
# Build final ResNet34

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


final_resnet34_model = models.resnet34(
    weights=ResNet34_Weights.IMAGENET1K_V1
)

for parameter in final_resnet34_model.parameters():
    parameter.requires_grad = False


in_features = (
    final_resnet34_model.fc.in_features
)

final_resnet34_model.fc = nn.Linear(
    in_features,
    2
)

final_resnet34_model = final_resnet34_model.to(
    DEVICE
)


final_resnet34_criterion = (
    nn.CrossEntropyLoss()
)


final_resnet34_optimizer = optim.AdamW(
    final_resnet34_model.fc.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


FINAL_RESNET34_EPOCHS = (
    best_epoch
)


print("Final ResNet34 configuration")
print("Device:", DEVICE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Augmentation: False")
print(
    "Classification threshold:",
    RESNET34_THRESHOLD
)
print(
    "Training epochs:",
    FINAL_RESNET34_EPOCHS
)


In [ ]:
# Train final ResNet34 with checkpointing

FINAL_CHECKPOINT_PATH = (
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_checkpoint.pt"
)

FINAL_HISTORY_PATH = (
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_training_history.csv"
)


final_resnet34_history = []

training_start = time.time()


for epoch in range(
    1,
    FINAL_RESNET34_EPOCHS + 1
):

    epoch_start = time.time()


    train_loss, train_f1 = train_one_epoch(
        final_resnet34_model,
        final_train_loader,
        final_resnet34_criterion,
        final_resnet34_optimizer
    )


    final_resnet34_history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_f1":
                train_f1
        }
    )


    # Save training history after every epoch
    final_resnet34_history_df = pd.DataFrame(
        final_resnet34_history
    )

    final_resnet34_history_df.to_csv(
        FINAL_HISTORY_PATH,
        index=False
    )


    # Save checkpoint after every epoch
    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                final_resnet34_model.state_dict(),

            "optimizer_state_dict":
                final_resnet34_optimizer.state_dict(),

            "architecture":
                "ResNet34",

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "augmentation":
                False,

            "classification_threshold":
                RESNET34_THRESHOLD,

            "random_seed":
                RANDOM_SEED
        },

        FINAL_CHECKPOINT_PATH
    )


    epoch_minutes = (
        time.time()
        - epoch_start
    ) / 60


    elapsed_minutes = (
        time.time()
        - training_start
    ) / 60


    average_epoch_minutes = (
        elapsed_minutes
        / epoch
    )


    remaining_epochs = (
        FINAL_RESNET34_EPOCHS
        - epoch
    )


    estimated_remaining_minutes = (
        average_epoch_minutes
        * remaining_epochs
    )


    print(
        f"\nEpoch {epoch:02d}/{FINAL_RESNET34_EPOCHS}"
    )

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f}"
    )

    print(
        f"Epoch time: {epoch_minutes:.2f} min | "
        f"Elapsed: {elapsed_minutes:.2f} min | "
        f"Estimated remaining: "
        f"{estimated_remaining_minutes:.2f} min"
    )

    print(
        "Checkpoint saved."
    )


total_training_minutes = (
    time.time()
    - training_start
) / 60


print(
    "\nFinal ResNet34 training complete"
)

print(
    "Total training time:",
    f"{total_training_minutes:.2f} minutes"
)

print(
    "Final checkpoint:",
    FINAL_CHECKPOINT_PATH
)


## 8. Held-out cross-site evaluation

The final ResNet34 model is evaluated on Cornwall Bude Cedar Grove and
Brutondam using the validation-selected threshold.

Performance is reported both across the pooled 800-image test set and
separately for each camera site. ROC-AUC measures score ranking independently
of the operating threshold, whereas accuracy, precision, recall and F1 describe
performance at the selected deployment threshold.

Per-site reporting is important because a pooled metric can conceal substantial
variation in model transfer between monitoring environments.


In [ ]:
# Final held-out evaluation

final_resnet34_model.eval()

all_targets = []
all_predictions = []
all_scores = []
all_sites = []
all_paths = []


with torch.no_grad():

    for images, labels, sites, paths in final_test_loader:

        images = images.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )

        logits = final_resnet34_model(
            images
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        predictions = (
            probabilities
            >= RESNET34_THRESHOLD
        ).long()


        all_targets.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_scores.extend(
            probabilities.cpu().numpy()
        )

        all_sites.extend(
            sites
        )

        all_paths.extend(
            paths
        )


resnet34_final_results_df = pd.DataFrame({

    "site":
        all_sites,

    "image_path":
        all_paths,

    "true_label":
        all_targets,

    "prediction":
        all_predictions,

    "blocked_probability":
        all_scores
})


resnet34_pooled_metrics = {

    "auc":
        roc_auc_score(
            resnet34_final_results_df["true_label"],
            resnet34_final_results_df["blocked_probability"]
        ),

    "accuracy":
        accuracy_score(
            resnet34_final_results_df["true_label"],
            resnet34_final_results_df["prediction"]
        ),

    "precision":
        precision_score(
            resnet34_final_results_df["true_label"],
            resnet34_final_results_df["prediction"],
            zero_division=0
        ),

    "recall":
        recall_score(
            resnet34_final_results_df["true_label"],
            resnet34_final_results_df["prediction"],
            zero_division=0
        ),

    "f1":
        f1_score(
            resnet34_final_results_df["true_label"],
            resnet34_final_results_df["prediction"],
            zero_division=0
        )
}


print(
    "Final classification threshold:",
    f"{RESNET34_THRESHOLD:.3f}"
)

print(
    "\nFinal pooled held-out performance"
)

for metric, value in resnet34_pooled_metrics.items():

    print(
        f"{metric}: {value:.4f}"
    )


resnet34_final_results_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_predictions.csv",
    index=False
)


In [ ]:
# Final ResNet34 per-site metrics

SITE_DISPLAY_NAMES = {
    "Cornwall_BudeCedarGrove":
        "Cornwall Bude Cedar Grove",

    "sites_brutondam_cam1":
        "Brutondam"
}


site_rows = []


for site in TEST_SITES:

    site_df = resnet34_final_results_df[
        resnet34_final_results_df["site"] == site
    ]

    site_rows.append(
        {
            "site":
                SITE_DISPLAY_NAMES.get(
                    site,
                    site
                ),

            "auc":
                roc_auc_score(
                    site_df["true_label"],
                    site_df["blocked_probability"]
                ),

            "accuracy":
                accuracy_score(
                    site_df["true_label"],
                    site_df["prediction"]
                ),

            "precision":
                precision_score(
                    site_df["true_label"],
                    site_df["prediction"],
                    zero_division=0
                ),

            "recall":
                recall_score(
                    site_df["true_label"],
                    site_df["prediction"],
                    zero_division=0
                ),

            "f1":
                f1_score(
                    site_df["true_label"],
                    site_df["prediction"],
                    zero_division=0
                )
        }
    )


resnet34_site_metrics_df = pd.DataFrame(
    site_rows
)


display(
    resnet34_site_metrics_df
)


resnet34_site_metrics_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_per_site_metrics.csv",
    index=False
)


In [ ]:
# Final ResNet34 confusion matrices

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)


# Pooled
pooled_cm = confusion_matrix(
    resnet34_final_results_df["true_label"],
    resnet34_final_results_df["prediction"]
)

ConfusionMatrixDisplay(
    confusion_matrix=pooled_cm,
    display_labels=[
        "clear",
        "blocked"
    ]
).plot(
    ax=axes[0],
    cmap="Blues",
    colorbar=False
)

axes[0].set_title(
    "Pooled"
)


# Individual test sites
for ax, site in zip(
    axes[1:],
    TEST_SITES
):

    site_df = resnet34_final_results_df[
        resnet34_final_results_df["site"]
        == site
    ]

    site_cm = confusion_matrix(
        site_df["true_label"],
        site_df["prediction"]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=site_cm,
        display_labels=[
            "clear",
            "blocked"
        ]
    ).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(
        SITE_DISPLAY_NAMES.get(
            site,
            site
        )
    )


fig.suptitle(
    "Final ResNet34 performance on unseen test sites",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Final ResNet34 ROC curves

plt.figure(
    figsize=(9, 6)
)


# Pooled ROC
pooled_fpr, pooled_tpr, _ = roc_curve(
    resnet34_final_results_df["true_label"],
    resnet34_final_results_df["blocked_probability"]
)

pooled_auc = roc_auc_score(
    resnet34_final_results_df["true_label"],
    resnet34_final_results_df["blocked_probability"]
)

plt.plot(
    pooled_fpr,
    pooled_tpr,
    label=f"Pooled (AUC = {pooled_auc:.3f})"
)


# Site-specific ROC curves
for site in TEST_SITES:

    site_df = resnet34_final_results_df[
        resnet34_final_results_df["site"] == site
    ]

    site_fpr, site_tpr, _ = roc_curve(
        site_df["true_label"],
        site_df["blocked_probability"]
    )

    site_auc = roc_auc_score(
        site_df["true_label"],
        site_df["blocked_probability"]
    )

    display_name = SITE_DISPLAY_NAMES.get(
        site,
        site
    )

    plt.plot(
        site_fpr,
        site_tpr,
        label=(
            f"{display_name} "
            f"(AUC = {site_auc:.3f})"
        )
    )


# Chance line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance"
)


plt.xlabel(
    "False positive rate"
)

plt.ylabel(
    "True positive rate"
)

plt.title(
    "ResNet34 performance on unseen test sites"
)

plt.legend()

plt.tight_layout()


plt.savefig(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_ROC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ResNet34 final results summary

resnet34_summary_df = pd.DataFrame(
    [
        {
            "model": "ResNet34",
            "site": "Pooled",
            "auc": resnet34_pooled_metrics["auc"],
            "accuracy": resnet34_pooled_metrics["accuracy"],
            "precision": resnet34_pooled_metrics["precision"],
            "recall": resnet34_pooled_metrics["recall"],
            "f1": resnet34_pooled_metrics["f1"]
        },

        {
            "model": "ResNet34",
            "site": "Cornwall Bude Cedar Grove",
            "auc": 0.771025,
            "accuracy": 0.5475,
            "precision": 1.0000,
            "recall": 0.0950,
            "f1": 0.173516
        },

        {
            "model": "ResNet34",
            "site": "Brutondam",
            "auc": 0.665600,
            "accuracy": 0.6200,
            "precision": 0.70339,
            "recall": 0.4150,
            "f1": 0.522013
        }
    ]
)

display(
    resnet34_summary_df.round(4)
)

resnet34_summary_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_results_summary.csv",
    index=False
)


## 9. Comparison with the ResNet50 reference model

The notebook optionally loads the corresponding ResNet50 result from
the repository outputs directory and compares both CNN baselines under the same
held-out site framework.

In [ ]:
# Compare ResNet50 and ResNet34

RESNET50_FOLDER = (
    REPO_ROOT / "outputs" / "resnet50_supervised"
)

RESNET50_RESULTS_PATH = (
    str(RESNET50_FOLDER) + "/"
    "ResNet50_final_results.json"
)


with open(
    RESNET50_RESULTS_PATH,
    "r"
) as file:

    resnet50_saved = json.load(
        file
    )


resnet50_pooled = (
    resnet50_saved[
        "test_results"
    ]
)

resnet50_site_results = (
    resnet50_saved[
        "test_results_by_site"
    ]
)


comparison_rows = []


# ResNet50 pooled
comparison_rows.append(
    {
        "model":
            "ResNet50",

        "site":
            "Pooled",

        "auc":
            resnet50_pooled["auc"],

        "accuracy":
            resnet50_pooled["accuracy"],

        "precision":
            resnet50_pooled["precision"],

        "recall":
            resnet50_pooled["recall"],

        "f1":
            resnet50_pooled["f1"]
    }
)


# ResNet50 per-site
for row in resnet50_site_results:

    site_name = SITE_DISPLAY_NAMES.get(
        row["site"],
        row["site"]
    )

    comparison_rows.append(
        {
            "model":
                "ResNet50",

            "site":
                site_name,

            "auc":
                row["auc"],

            "accuracy":
                row["accuracy"],

            "precision":
                row["precision"],

            "recall":
                row["recall"],

            "f1":
                row["f1"]
        }
    )


# ResNet34 pooled
comparison_rows.append(
    {
        "model":
            "ResNet34",

        "site":
            "Pooled",

        "auc":
            resnet34_pooled_metrics["auc"],

        "accuracy":
            resnet34_pooled_metrics["accuracy"],

        "precision":
            resnet34_pooled_metrics["precision"],

        "recall":
            resnet34_pooled_metrics["recall"],

        "f1":
            resnet34_pooled_metrics["f1"]
    }
)


# ResNet34 per-site
for _, row in resnet34_site_metrics_df.iterrows():

    comparison_rows.append(
        {
            "model":
                "ResNet34",

            "site":
                row["site"],

            "auc":
                row["auc"],

            "accuracy":
                row["accuracy"],

            "precision":
                row["precision"],

            "recall":
                row["recall"],

            "f1":
                row["f1"]
        }
    )


resnet_comparison_df = pd.DataFrame(
    comparison_rows
)


resnet_comparison_df = (
    resnet_comparison_df[
        [
            "model",
            "site",
            "auc",
            "accuracy",
            "precision",
            "recall",
            "f1"
        ]
    ]
)


display(
    resnet_comparison_df.round(4)
)


resnet_comparison_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_vs_ResNet50_comparison.csv",
    index=False
)


### Recorded dissertation comparison: ResNet34 and ResNet50

ResNet50 achieved stronger performance than ResNet34 across both unseen test sites.
At the pooled level, ResNet50 achieved an AUC of 0.820 and F1 score of 0.751,
compared with an AUC of 0.689 and F1 score of 0.380 for ResNet34.

The largest difference was observed in recall. ResNet50 identified 69.0% of blocked
images across the held-out sites, whereas ResNet34 identified 25.5%. The difference
was particularly pronounced at Cornwall Bude Cedar Grove, where ResNet34 achieved
a recall of only 9.5%.

Although ResNet34 achieved a precision of 1.000 at Cornwall Bude Cedar Grove, this
result reflects the small number of images classified as blocked rather than stronger
overall performance. Its recall of 0.095 shows that most blocked images at this site
were classified as clear.

Under the same site-based evaluation framework, the deeper ResNet50 architecture
therefore provided substantially better generalisation to the two unseen camera sites.
These values describe the recorded dissertation run. The public notebook has cleared outputs, so rerunning the experiment will generate a new execution record from the same documented procedure.


## 10. Save reproducibility artefacts

The final cells export the trained model state, development and final training
histories, validation threshold search, image-level predictions, per-site
metrics, diagnostic figures and summary results.

These artefacts allow the reported experiment to be inspected independently of
the transient state of the notebook.


In [ ]:
# Save final ResNet34 results

FINAL_RESNET34_RESULTS_PATH = (
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_final_results.json"
)


resnet34_final_results = {

    "model": {
        "architecture":
            "ResNet34",

        "pretrained_weights":
            "ImageNet1K_V1",

        "backbone":
            "frozen",

        "batchnorm_statistics":
            "fixed",

        "trainable_layer":
            "final_fc_layer",

        "classes":
            2
    },


    "training": {
        "optimizer":
            "AdamW",

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            float(
                WEIGHT_DECAY
            ),

        "augmentation":
            False,

        "epochs":
            int(
                FINAL_RESNET34_EPOCHS
            ),

        "batch_size":
            int(
                BATCH_SIZE
            ),

        "random_seed":
            int(
                RANDOM_SEED
            )
    },


    "selection": {
        "best_development_epoch":
            int(
                best_epoch
            ),

        "best_development_f1":
            float(
                best_validation_f1
            ),

        "classification_threshold":
            float(
                RESNET34_THRESHOLD
            ),

        "threshold_selected_using":
            "validation_f1"
    },


    "data": {
        "development_sites":
            list(
                DEVELOPMENT_SITES
            ),

        "training_sites":
            list(
                TRAIN_SITES
            ),

        "validation_sites":
            list(
                VALIDATION_SITES
            ),

        "test_sites":
            list(
                TEST_SITES
            ),

        "final_training_images":
            int(
                len(final_train_df)
            ),

        "final_test_images":
            int(
                len(final_test_df)
            )
    },


    "test_results": {
        "auc":
            float(
                resnet34_pooled_metrics["auc"]
            ),

        "accuracy":
            float(
                resnet34_pooled_metrics["accuracy"]
            ),

        "precision":
            float(
                resnet34_pooled_metrics["precision"]
            ),

        "recall":
            float(
                resnet34_pooled_metrics["recall"]
            ),

        "f1":
            float(
                resnet34_pooled_metrics["f1"]
            )
    },


    "test_results_by_site":
        resnet34_site_metrics_df.to_dict(
            orient="records"
        )
}


with open(
    FINAL_RESNET34_RESULTS_PATH,
    "w"
) as file:

    json.dump(
        resnet34_final_results,
        file,
        indent=4
    )


resnet_comparison_df.to_csv(
    str(EXPERIMENT_FOLDER) + "/"
    "ResNet34_vs_ResNet50_comparison.csv",
    index=False
)


print(
    "Final ResNet34 results saved:"
)

print(
    FINAL_RESNET34_RESULTS_PATH
)

print(
    "\nResNet34 vs ResNet50 comparison saved."
)


In [ ]:
# Check saved ResNet34 files

files_to_check = [

    FINAL_CHECKPOINT_PATH,

    FINAL_RESNET34_RESULTS_PATH,

    FINAL_HISTORY_PATH,

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_development_history.csv"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_validation_threshold_search.csv"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_final_predictions.csv"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_final_per_site_metrics.csv"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_final_confusion_matrices.png"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_final_ROC.png"
    ),

    (
        str(EXPERIMENT_FOLDER) + "/"
        "ResNet34_vs_ResNet50_comparison.csv"
    )
]


check_rows = []


for path in files_to_check:

    check_rows.append(
        {
            "file":
                os.path.basename(
                    path
                ),

            "exists":
                os.path.exists(
                    path
                )
        }
    )


saved_files_df = pd.DataFrame(
    check_rows
)


display(
    saved_files_df
)


if saved_files_df["exists"].all():

    print(
        "All important ResNet34 files are saved."
    )

else:

    print(
        "One or more files are missing."
    )
